In [17]:
import pandas as pd
import numpy as np

# ========= 1. Reading Data =========
file_path = "AoA_51715_words.xlsx"   
df = pd.read_excel(file_path, sheet_name="Sheet1")

print("The number of rows of raw data:", len(df))
print("The number of column of raw data:", df.columns.tolist())


# ========= 2. Basic cleaning =========
# Remove empty lines in Word
df = df.dropna(subset=["Word"]).copy()

# Deduplication: If the same word is repeated, keep the first one.
df = df.drop_duplicates(subset=["Word"]).copy()

# Convert any potentially used numeric columns to numeric to avoid string interference.
numeric_cols = [
    "Freq_pm", "Nletters", "Nphon", "Nsyll",
    "AoA_Kup", "Perc_known", "AoA_Kup_lem", "Perc_known_lem",
    "AoA_Bird_lem", "AoA_Bristol_lem", "AoA_Cort_lem", "AoA_Schock"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# ========= 3. Filtering low-quality samples =========
# First, keep the words with a Perc_known value greater than or equal to 0.8.
# If Perc_known is missing, try using Perc_known_lem
df["Perc_known_final"] = df["Perc_known"].combine_first(df["Perc_known_lem"])

df_clean = df[df["Perc_known_final"] >= 0.8].copy()

print("\nPerc_known Number of rows after filtering:", len(df_clean))


# ========= 4. Synthetic AoA_final =========
# Priority：
# AoA_Kup -> AoA_Kup_lem -> AoA_Bird_lem -> AoA_Bristol_lem -> AoA_Cort_lem -> AoA_Schock
df_clean["AoA_final"] = (
    df_clean["AoA_Kup"]
    .combine_first(df_clean["AoA_Kup_lem"])
    .combine_first(df_clean["AoA_Bird_lem"])
    .combine_first(df_clean["AoA_Bristol_lem"])
    .combine_first(df_clean["AoA_Cort_lem"])
    .combine_first(df_clean["AoA_Schock"])
)

# Remove words without any AoA tags.
df_clean = df_clean.dropna(subset=["AoA_final"]).copy()

print("Complete the number of rows after AoA:", len(df_clean))


# ========= 5. By age group =========
def assign_age_group(x):
    if pd.isna(x):
        return np.nan
    elif x < 6:
        return "below_6"
    elif 6 <= x < 8:
        return "age_6_8"
    else:
        return "age_8_plus"

df_clean["Age_Group"] = df_clean["AoA_final"].apply(assign_age_group)


# ========= 6. Statistical distribution =========
dist = df_clean["Age_Group"].value_counts(dropna=False)
dist_pct = df_clean["Age_Group"].value_counts(normalize=True, dropna=False) * 100

summary_df = pd.DataFrame({
    "count": dist,
    "percentage": dist_pct.round(2)
})

print("\nAge group distribution：")
print(summary_df)


# ========= 7. See each set of examples =========
print("\nbelow_6 example：")
print(df_clean[df_clean["Age_Group"] == "below_6"][["Word", "AoA_final"]].head(10))

print("\nage_6_8 example：")
print(df_clean[df_clean["Age_Group"] == "age_6_8"][["Word", "AoA_final"]].head(10))

print("\nage_8_plus example：")
print(df_clean[df_clean["Age_Group"] == "age_8_plus"][["Word", "AoA_final"]].head(10))


# ========= 8. Output results =========
# Output complete cleaning data
output_clean = "AoA_refined_dataset.xlsx"

with pd.ExcelWriter(output_clean, engine="openpyxl") as writer:
    df_clean.to_excel(writer, sheet_name="refined_data", index=False)
    summary_df.to_excel(writer, sheet_name="distribution")

df_clean[df_clean["Age_Group"] == "below_6"].to_excel(output_below6, index=False)
df_clean[df_clean["Age_Group"] == "age_6_8"].to_excel(output_6_8, index=False)
df_clean[df_clean["Age_Group"] == "age_8_plus"].to_excel(output_8_plus, index=False)

print("\nThe file has been output.：")
print(output_clean)

The number of rows of raw data: 51715
The number of column of raw data: ['Word', 'Alternative.spelling', 'Freq_pm', 'Dom_PoS_SUBTLEX', 'Nletters', 'Nphon', 'Nsyll', 'Lemma_highest_PoS', 'AoA_Kup', 'Perc_known', 'AoA_Kup_lem', 'Perc_known_lem', 'AoA_Bird_lem', 'AoA_Bristol_lem', 'AoA_Cort_lem', 'AoA_Schock']

Perc_known Number of rows after filtering: 42831
Complete the number of rows after AoA: 42831

Age group distribution：
            count  percentage
Age_Group                    
age_8_plus  30821       71.96
age_6_8      6982       16.30
below_6      5028       11.74

below_6 example：
          Word  AoA_final
0            a   2.893384
109      about   5.070761
110      above   4.548568
256   accident   5.300000
259  accidents   5.300000
357       ache   5.790000
358      ached   5.790000
359      aches   5.790000
393      acorn   5.950000
394     acorns   5.950000

age_6_8 example：
            Word  AoA_final
69          able   7.790000
70         abler   7.790000
71        ables

In [19]:
check_cols = [
    "Freq_pm", "Dom_PoS_SUBTLEX",
    "AoA_Kup", "AoA_Kup_lem",
    "AoA_Bird_lem", "AoA_Bristol_lem",
    "AoA_Cort_lem", "AoA_Schock",
    "AoA_final"
]

missing_summary = pd.DataFrame({
    "missing_count": df_clean[check_cols].isna().sum(),
    "missing_pct": (df_clean[check_cols].isna().mean() * 100).round(2)
})

print(missing_summary)

                 missing_count  missing_pct
Freq_pm                    135         0.32
Dom_PoS_SUBTLEX            185         0.43
AoA_Kup                  18851        44.01
AoA_Kup_lem                  0         0.00
AoA_Bird_lem             37511        87.58
AoA_Bristol_lem          34966        81.64
AoA_Cort_lem             35036        81.80
AoA_Schock               35906        83.83
AoA_final                    0         0.00


In [21]:
import numpy as np

def get_aoa_source(row):
    if pd.notna(row["AoA_Kup"]):
        return "AoA_Kup"
    elif pd.notna(row["AoA_Kup_lem"]):
        return "AoA_Kup_lem"
    elif pd.notna(row["AoA_Bird_lem"]):
        return "AoA_Bird_lem"
    elif pd.notna(row["AoA_Bristol_lem"]):
        return "AoA_Bristol_lem"
    elif pd.notna(row["AoA_Cort_lem"]):
        return "AoA_Cort_lem"
    elif pd.notna(row["AoA_Schock"]):
        return "AoA_Schock"
    else:
        return np.nan

df_clean["AoA_source"] = df_clean.apply(get_aoa_source, axis=1)

source_summary = pd.DataFrame({
    "count": df_clean["AoA_source"].value_counts(),
    "percentage": (df_clean["AoA_source"].value_counts(normalize=True) * 100).round(2)
})

print(source_summary)

             count  percentage
AoA_source                    
AoA_Kup      23980       55.99
AoA_Kup_lem  18851       44.01


In [23]:
df_v2 = df_clean.copy()

# Fill in the missing parts of speech
df_v2["Dom_PoS_SUBTLEX"] = df_v2["Dom_PoS_SUBTLEX"].fillna("Unknown")

# Filtering low-frequency words
df_v2 = df_v2[df_v2["Freq_pm"].notna()]
df_v2 = df_v2[df_v2["Freq_pm"] >= 1].copy()

print("v2 Number of data rows:", len(df_v2))

v2_dist = pd.DataFrame({
    "count": df_v2["Age_Group"].value_counts(),
    "percentage": (df_v2["Age_Group"].value_counts(normalize=True) * 100).round(2)
})

print("\nv2 Age group distribution：")
print(v2_dist)

v2 Number of data rows: 14471

v2 Age group distribution：
            count  percentage
Age_Group                    
age_8_plus   7380       51.00
age_6_8      3592       24.82
below_6      3499       24.18


In [25]:
with pd.ExcelWriter("AoA_refined_dataset_v2.xlsx", engine="openpyxl") as writer:
    df_v2.to_excel(writer, sheet_name="refined_v2", index=False)
    source_summary.to_excel(writer, sheet_name="AoA_source_summary")
    v2_dist.to_excel(writer, sheet_name="Age_group_distribution")

print("Output completed: AoA_refined_dataset_v2.xlsx")

Output completed: AoA_refined_dataset_v2.xlsx


In [27]:
for threshold in [1, 2, 5]:
    temp = df_clean.copy()
    temp["Dom_PoS_SUBTLEX"] = temp["Dom_PoS_SUBTLEX"].fillna("Unknown")
    temp = temp[temp["Freq_pm"].notna()]
    temp = temp[temp["Freq_pm"] >= threshold].copy()

    dist = pd.DataFrame({
        "count": temp["Age_Group"].value_counts(),
        "percentage": (temp["Age_Group"].value_counts(normalize=True) * 100).round(2)
    })

    print(f"\nFreq_pm >= {threshold}")
    print("rows:", len(temp))
    print(dist)


Freq_pm >= 1
rows: 14471
            count  percentage
Age_Group                    
age_8_plus   7380       51.00
age_6_8      3592       24.82
below_6      3499       24.18

Freq_pm >= 2
rows: 9997
            count  percentage
Age_Group                    
age_8_plus   4245       42.46
below_6      3035       30.36
age_6_8      2717       27.18

Freq_pm >= 5
rows: 5889
            count  percentage
Age_Group                    
below_6      2324       39.46
age_8_plus   1830       31.07
age_6_8      1735       29.46


In [29]:
df_v3_freq2 = df_clean.copy()

# Fill in the missing parts of speech
df_v3_freq2["Dom_PoS_SUBTLEX"] = df_v3_freq2["Dom_PoS_SUBTLEX"].fillna("Unknown")

# Keep word frequency and word frequency > = 2
df_v3_freq2 = df_v3_freq2[df_v3_freq2["Freq_pm"].notna()].copy()
df_v3_freq2 = df_v3_freq2[df_v3_freq2["Freq_pm"] >= 2].copy()

# Statistical age distribution
v3_freq2_dist = pd.DataFrame({
    "count": df_v3_freq2["Age_Group"].value_counts(),
    "percentage": (df_v3_freq2["Age_Group"].value_counts(normalize=True) * 100).round(2)
})

print("Freq_pm >= 2 Number of rows of data:", len(df_v3_freq2))
print(v3_freq2_dist)

# Export Excel
with pd.ExcelWriter("AoA_refined_dataset_v3_freq2.xlsx", engine="openpyxl") as writer:
    df_v3_freq2.to_excel(writer, sheet_name="refined_freq2", index=False)
    source_summary.to_excel(writer, sheet_name="AoA_source_summary")
    v3_freq2_dist.to_excel(writer, sheet_name="Age_group_distribution")

print("Output: AoA_refined_dataset_v3_freq2.xlsx")

Freq_pm >= 2 Number of rows of data: 9997
            count  percentage
Age_Group                    
age_8_plus   4245       42.46
below_6      3035       30.36
age_6_8      2717       27.18
Output: AoA_refined_dataset_v3_freq2.xlsx


In [31]:
df_v3_freq5 = df_clean.copy()

# Fill in the missing parts of speech
df_v3_freq5["Dom_PoS_SUBTLEX"] = df_v3_freq5["Dom_PoS_SUBTLEX"].fillna("Unknown")

# Keep word frequency and word frequency > = 5
df_v3_freq5 = df_v3_freq5[df_v3_freq5["Freq_pm"].notna()].copy()
df_v3_freq5 = df_v3_freq5[df_v3_freq5["Freq_pm"] >= 5].copy()

# Statistical age distribution
v3_freq5_dist = pd.DataFrame({
    "count": df_v3_freq5["Age_Group"].value_counts(),
    "percentage": (df_v3_freq5["Age_Group"].value_counts(normalize=True) * 100).round(2)
})

print("Freq_pm >= 5 Number of rows of data:", len(df_v3_freq5))
print(v3_freq5_dist)

# Export Excel
with pd.ExcelWriter("AoA_refined_dataset_v3_freq5.xlsx", engine="openpyxl") as writer:
    df_v3_freq5.to_excel(writer, sheet_name="refined_freq5", index=False)
    source_summary.to_excel(writer, sheet_name="AoA_source_summary")
    v3_freq5_dist.to_excel(writer, sheet_name="Age_group_distribution")

print("Output: AoA_refined_dataset_v3_freq5.xlsx")

Freq_pm >= 5 Number of rows of data: 5889
            count  percentage
Age_Group                    
below_6      2324       39.46
age_8_plus   1830       31.07
age_6_8      1735       29.46
Output: AoA_refined_dataset_v3_freq5.xlsx
